## 1. Import Libraries

The following Python libraries are used for data manipulation, numerical calculations, and feature engineering.

- Pandas for data manipulation
- NumPy for numerical operations
- Matplotlib for basic visualization when required

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Load the Analysis Dataset

The feature engineering process starts with the cleaned and integrated FORESIGHT dataset prepared during the previous stages.

The `analysis_ready.csv` file contains the sales, SKU, calendar, and inventory-related information required for downstream feature engineering.

In [2]:
# Load the analysis-ready dataset

df = pd.read_csv("../data/processed/analysis_ready.csv")

print("Dataset loaded successfully.")
print("Shape:", df.shape)

Dataset loaded successfully.
Shape: (20906, 26)


In [3]:
# Display the available columns

print("Available columns:")
print(df.columns.tolist())

Available columns:
['date', 'sku_id', 'units_sold', 'revenue', 'price', 'promo_flag', 'calculated_revenue', 'revenue_difference', 'revenue_consistency_flag', 'category', 'subcategory', 'launch_date', 'unit_cost', 'list_price', 'sku_quality_flag', 'week', 'month', 'season', 'is_holiday', 'promo_event', 'promo_flag_calendar', 'on_hand_units', 'on_order_units', 'lead_time_days', 'reorder_point', 'inventory_quality_flag']


In [4]:
# Convert date column to datetime format

df["date"] = pd.to_datetime(df["date"], errors="coerce")

print("Date conversion completed.")
print("Date range:", df["date"].min(), "to", df["date"].max())

Date conversion completed.
Date range: 2025-01-01 00:00:00 to 2026-06-30 00:00:00


In [5]:
# Create calendar-based features

df["day_of_week"] = df["date"].dt.dayofweek
df["day_of_month"] = df["date"].dt.day
df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)
df["month"] = df["date"].dt.month
df["quarter"] = df["date"].dt.quarter
df["year"] = df["date"].dt.year
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

print("Date features created successfully.")

Date features created successfully.


In [6]:
# Check the newly created date features

date_features = [
    "date",
    "day_of_week",
    "day_of_month",
    "week_of_year",
    "month",
    "quarter",
    "year",
    "is_weekend"
]

df[date_features].head(10)

,date,day_of_week,day_of_month,week_of_year,month,quarter,year,is_weekend
0,2025-01-01,2,1,1,1,1,2025,0
1,2025-01-02,3,2,1,1,1,2025,0
2,2025-01-03,4,3,1,1,1,2025,0
3,2025-01-04,5,4,1,1,1,2025,1
4,2025-01-05,6,5,1,1,1,2025,1
5,2025-01-06,0,6,2,1,1,2025,0
6,2025-01-07,1,7,2,1,1,2025,0
7,2025-01-08,2,8,2,1,1,2025,0
8,2025-01-09,3,9,2,1,1,2025,0
9,2025-01-10,4,10,2,1,1,2025,0


In [7]:
# Create lag features for historical demand

df = df.sort_values(["sku_id", "date"]).copy()

df["lag_1"] = (
    df.groupby("sku_id")["units_sold"]
      .shift(1)
)

df["lag_7"] = (
    df.groupby("sku_id")["units_sold"]
      .shift(7)
)

df["lag_14"] = (
    df.groupby("sku_id")["units_sold"]
      .shift(14)
)

df["lag_28"] = (
    df.groupby("sku_id")["units_sold"]
      .shift(28)
)

print("Lag features created successfully.")

Lag features created successfully.


In [8]:
# Check the newly created lag features

lag_features = [
    "date",
    "sku_id",
    "units_sold",
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28"
]

df[lag_features].head(15)

,date,sku_id,units_sold,lag_1,lag_7,lag_14,lag_28
0,2025-01-01,SKU001,52.0,NaN,NaN,NaN,NaN
1,2025-01-02,SKU001,69.0,52.0,NaN,NaN,NaN
2,2025-01-03,SKU001,49.0,69.0,NaN,NaN,NaN
3,2025-01-04,SKU001,74.0,49.0,NaN,NaN,NaN
4,2025-01-05,SKU001,43.0,74.0,NaN,NaN,NaN
5,2025-01-06,SKU001,56.0,43.0,NaN,NaN,NaN
6,2025-01-07,SKU001,46.0,56.0,NaN,NaN,NaN
7,2025-01-08,SKU001,51.0,46.0,52.0,NaN,NaN
8,2025-01-09,SKU001,65.0,51.0,69.0,NaN,NaN
9,2025-01-10,SKU001,70.0,65.0,49.0,NaN,NaN


In [9]:
# Create rolling demand features

df["rolling_mean_7"] = (
    df.groupby("sku_id")["units_sold"]
      .transform(lambda x: x.shift(1).rolling(window=7).mean())
)

df["rolling_mean_14"] = (
    df.groupby("sku_id")["units_sold"]
      .transform(lambda x: x.shift(1).rolling(window=14).mean())
)

df["rolling_mean_28"] = (
    df.groupby("sku_id")["units_sold"]
      .transform(lambda x: x.shift(1).rolling(window=28).mean())
)

print("Rolling demand features created successfully.")

Rolling demand features created successfully.


In [10]:
# Create rolling demand volatility features

df["rolling_std_7"] = (
    df.groupby("sku_id")["units_sold"]
      .transform(lambda x: x.shift(1).rolling(window=7).std())
)

df["rolling_std_14"] = (
    df.groupby("sku_id")["units_sold"]
      .transform(lambda x: x.shift(1).rolling(window=14).std())
)

df["rolling_std_28"] = (
    df.groupby("sku_id")["units_sold"]
      .transform(lambda x: x.shift(1).rolling(window=28).std())
)

print("Rolling demand volatility features created successfully.")

Rolling demand volatility features created successfully.


In [11]:
# Create a demand trend feature

df["demand_trend_7_28"] = (
    df["rolling_mean_7"] - df["rolling_mean_28"]
)

print("Demand trend feature created successfully.")

Demand trend feature created successfully.


In [12]:
# Create promotion-related features

if "promo_flag" in df.columns:
    
    df["promo_flag"] = df["promo_flag"].fillna(0).astype(int)
    
    df["promo_demand_interaction"] = (
        df["promo_flag"] * df["rolling_mean_7"]
    )
    
    print("Promotion features created successfully.")

else:
    print("promo_flag column not found.")

Promotion features created successfully.


In [13]:
# Create price change features

if "price" in df.columns:
    
    df["price_change"] = (
        df.groupby("sku_id")["price"]
          .pct_change()
          .replace([np.inf, -np.inf], np.nan)
          .fillna(0)
    )
    
    print("Price change feature created successfully.")

else:
    print("price column not found.")

Price change feature created successfully.


In [14]:
# Create inventory-related features

inventory_columns = [
    "on_hand_units",
    "on_order_units",
    "lead_time_days",
    "reorder_point"
]

available_inventory_columns = [
    col for col in inventory_columns
    if col in df.columns
]

print("Available inventory columns:")
print(available_inventory_columns)

if "on_hand_units" in df.columns:
    
    df["stock_gap"] = (
        df["on_hand_units"] - df.get("reorder_point", 0)
    )
    
    df["stockout_flag"] = (
        df["on_hand_units"] <= 0
    ).astype(int)

if "on_order_units" in df.columns:
    
    df["total_available_stock"] = (
        df["on_hand_units"].fillna(0)
        + df["on_order_units"].fillna(0)
    )

print("Inventory features created successfully.")

Available inventory columns:
['on_hand_units', 'on_order_units', 'lead_time_days', 'reorder_point']
Inventory features created successfully.


In [15]:
# Create date-based features

df["date"] = pd.to_datetime(df["date"], errors="coerce")

df["day_of_week"] = df["date"].dt.dayofweek
df["day_of_month"] = df["date"].dt.day
df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)
df["month"] = df["date"].dt.month
df["quarter"] = df["date"].dt.quarter
df["year"] = df["date"].dt.year

df["is_weekend"] = (
    df["day_of_week"] >= 5
).astype(int)

print("Date features created successfully.")

Date features created successfully.


In [16]:
# Create historical demand lag features

df = df.sort_values(["sku_id", "date"]).reset_index(drop=True)

df["lag_1"] = (
    df.groupby("sku_id")["units_sold"]
      .shift(1)
)

df["lag_7"] = (
    df.groupby("sku_id")["units_sold"]
      .shift(7)
)

df["lag_14"] = (
    df.groupby("sku_id")["units_sold"]
      .shift(14)
)

df["lag_28"] = (
    df.groupby("sku_id")["units_sold"]
      .shift(28)
)

print("Lag features created successfully.")

Lag features created successfully.


In [17]:
# Create rolling demand features

df["rolling_mean_7"] = (
    df.groupby("sku_id")["units_sold"]
      .transform(
          lambda x: x.shift(1).rolling(7).mean()
      )
)

df["rolling_mean_14"] = (
    df.groupby("sku_id")["units_sold"]
      .transform(
          lambda x: x.shift(1).rolling(14).mean()
      )
)

df["rolling_mean_28"] = (
    df.groupby("sku_id")["units_sold"]
      .transform(
          lambda x: x.shift(1).rolling(28).mean()
      )
)

print("Rolling demand features created successfully.")

Rolling demand features created successfully.


In [18]:
# Create promotion and price-related features

if "promo_flag" in df.columns:
    df["promo_flag"] = df["promo_flag"].fillna(0).astype(int)

    df["promo_lag_1"] = (
        df.groupby("sku_id")["promo_flag"]
          .shift(1)
          .fillna(0)
    )

if "price" in df.columns:
    df["price_change"] = (
        df.groupby("sku_id")["price"]
          .pct_change()
          .replace([np.inf, -np.inf], np.nan)
          .fillna(0)
    )

print("Promotion and price features created successfully.")

Promotion and price features created successfully.


In [19]:
# Review the engineered dataset

print("Dataset shape after feature engineering:")
print(df.shape)

print("\nNew feature columns:")

feature_keywords = [
    "day_",
    "week_",
    "month",
    "quarter",
    "year",
    "weekend",
    "lag_",
    "rolling_",
    "stock_",
    "total_available",
    "promo_",
    "price_change"
]

engineered_columns = [
    col for col in df.columns
    if any(keyword in col for keyword in feature_keywords)
]

print(engineered_columns)

print("\nSample engineered data:")
display(
    df[
        [
            "date",
            "sku_id",
            "units_sold",
            "lag_1",
            "lag_7",
            "rolling_mean_7",
            "promo_flag",
            "price_change",
            "stockout_flag"
        ]
    ].head(10)
)

Dataset shape after feature engineering:
(20906, 49)

New feature columns:
['promo_flag', 'month', 'promo_event', 'promo_flag_calendar', 'day_of_week', 'day_of_month', 'week_of_year', 'quarter', 'year', 'is_weekend', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28', 'rolling_std_7', 'rolling_std_14', 'rolling_std_28', 'promo_demand_interaction', 'price_change', 'stock_gap', 'total_available_stock', 'promo_lag_1']

Sample engineered data:


,date,sku_id,units_sold,lag_1,lag_7,rolling_mean_7,promo_flag,price_change,stockout_flag
0,2025-01-01,SKU001,52.0,NaN,NaN,NaN,0,0.000000,0
1,2025-01-02,SKU001,69.0,52.0,NaN,NaN,0,0.000000,0
2,2025-01-03,SKU001,49.0,69.0,NaN,NaN,0,-0.013965,0
3,2025-01-04,SKU001,74.0,49.0,NaN,NaN,0,0.583795,0
4,2025-01-05,SKU001,43.0,74.0,NaN,NaN,0,-0.359663,0
5,2025-01-06,SKU001,56.0,43.0,NaN,NaN,0,-0.703106,0
6,2025-01-07,SKU001,46.0,56.0,NaN,NaN,0,2.368210,0
7,2025-01-08,SKU001,51.0,46.0,52.0,55.571429,0,0.366175,1
8,2025-01-09,SKU001,65.0,51.0,69.0,55.428571,0,0.000000,1
9,2025-01-10,SKU001,70.0,65.0,49.0,54.857143,1,-0.421647,1


In [20]:
# Check missing values after feature engineering

missing_features = (
    df.isnull()
      .sum()
      .sort_values(ascending=False)
)

missing_features = missing_features[
    missing_features > 0
]

print("Columns containing missing values:")
display(missing_features)

Columns containing missing values:


promo_event                 17634
rolling_mean_28              1498
rolling_std_28               1498
demand_trend_7_28            1498
lag_28                       1135
rolling_mean_14               756
rolling_std_14                756
lag_14                        575
unit_cost                     546
promo_demand_interaction      382
rolling_std_7                 382
rolling_mean_7                382
lag_7                         295
lag_1                          55
stock_gap                      40
on_hand_units                  40
calculated_revenue             25
revenue_difference             25
units_sold                     15
price                          10
dtype: int64

In [21]:
# Check for duplicate date-SKU records

duplicate_records = df.duplicated(
    subset=["date", "sku_id"]
).sum()

print("Duplicate date-SKU records:", duplicate_records)

Duplicate date-SKU records: 0


In [22]:
# Review data types of engineered features

print("Engineered dataset data types:")

display(
    df.dtypes.to_frame(name="data_type")
)

Engineered dataset data types:


,data_type
date,datetime64[us]
sku_id,str
units_sold,float64
revenue,float64
price,float64
promo_flag,int64
calculated_revenue,float64
revenue_difference,float64
revenue_consistency_flag,str
category,str


In [23]:
# Save the feature-engineered dataset

output_path = "../data/processed/feature_engineered.csv"

df.to_csv(
    output_path,
    index=False
)

print("Feature-engineered dataset saved successfully.")
print("File:", output_path)
print("Shape:", df.shape)

Feature-engineered dataset saved successfully.
File: ../data/processed/feature_engineered.csv
Shape: (20906, 49)


In [24]:
# Final summary of the feature-engineered dataset

print("FEATURE ENGINEERING SUMMARY")
print("=" * 50)

print("Total rows:", df.shape[0])
print("Total columns:", df.shape[1])
print("Unique SKUs:", df["sku_id"].nunique())
print("Date range:", df["date"].min(), "to", df["date"].max())

print("\nTarget variable:")
print("units_sold")

print("\nFeature-engineered dataset is ready for demand forecasting.")

FEATURE ENGINEERING SUMMARY
Total rows: 20906
Total columns: 49
Unique SKUs: 40
Date range: 2025-01-01 00:00:00 to 2026-06-30 00:00:00

Target variable:
units_sold

Feature-engineered dataset is ready for demand forecasting.
